In [6]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from forecast_checks import ForecastValidator, MakeTrainingData, DELAY_DAYS

In [7]:
SCORING_METRIC = "neg_root_mean_square_error"

In [8]:
trainer = MakeTrainingData()
data = trainer.get_train_test()
train_split = data["train"]
test_split=data["test"]
X_train, y_train = train_split["feature"], train_split["target"]
X_test, y_test = test_split["feature"], test_split["target"]

In [9]:
#Score every model on the same hours as neural_network.ipynb: drop test rows where a consumption lag is missing
scored = X_test.filter(like="consumption").notna().all(axis=1)
X_test, y_test = X_test[scored], y_test[scored]

In [10]:
#Hyperparameter tuning with CVGrid

NUM_TREES=[100,500,1000]
LEARNING_RATE=[0.025, 0.05, 0.075]
MAX_DEPTH=[5,10,15]

BASE_PARAMS = dict(enable_categorical=True, tree_method="hist", random_state=42)
param_grid = {"n_estimators": NUM_TREES, "learning_rate": LEARNING_RATE, "max_depth": MAX_DEPTH}
#Rows are sorted by time; the gap leaves out ~6 days of rows between fit and validation, like the consumption delay
cv = TimeSeriesSplit(n_splits=3, gap=(DELAY_DAYS + 1) * 24 * X_train["location"].nunique())
search = GridSearchCV(xgb.XGBRegressor(**BASE_PARAMS), param_grid, cv=cv,
                      scoring=SCORING_METRIC, refit=False,
                      verbose=True)
search.fit(X_train, y_train)
print("Best params:", search.best_params_)

InvalidParameterError: The 'scoring' parameter of GridSearchCV must be a str among {'average_precision', 'precision_macro', 'jaccard_weighted', 'r2', 'accuracy', 'mutual_info_score', 'recall_weighted', 'jaccard_micro', 'neg_root_mean_squared_log_error', 'roc_auc_ovr_weighted', 'jaccard_macro', 'roc_auc_ovo', 'f1_weighted', 'homogeneity_score', 'f1_samples', 'fowlkes_mallows_score', 'precision', 'neg_root_mean_squared_error', 'd2_log_loss_score', 'neg_mean_poisson_deviance', 'precision_samples', 'neg_mean_absolute_percentage_error', 'd2_absolute_error_score', 'precision_micro', 'balanced_accuracy', 'explained_variance', 'f1', 'recall_samples', 'neg_negative_likelihood_ratio', 'precision_weighted', 'd2_brier_score', 'recall_micro', 'recall_macro', 'neg_max_error', 'neg_mean_absolute_error', 'v_measure_score', 'recall', 'neg_brier_score', 'neg_mean_gamma_deviance', 'roc_auc_ovo_weighted', 'f1_macro', 'jaccard', 'jaccard_samples', 'matthews_corrcoef', 'neg_log_loss', 'positive_likelihood_r

In [ ]:
#TRAIN

model = xgb.XGBRegressor(**BASE_PARAMS, **search.best_params_)
model.fit(X_train, y_train)
pred = model.predict(X_test)

In [ ]:
#TEST

forecastValidator = (ForecastValidator(y_test, groups=X_test["location"])
                     .add("Same hour last week", X_test["consumption_7days_ago"], baseline=True)
                     .add("XGBoost", pred))
forecastValidator.summary()

In [ ]:
forecastValidator.save_to_json("results/xgboost.json")

In [ ]:
forecastValidator.per_group()

In [ ]:
forecastValidator.plot_predicted_vs_actual_consumption(no_baseline=True)

In [ ]:
forecastValidator.plot_consumption_over_time(trainer.df.loc[y_test.index, "time"], no_baseline=True)